In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# שימוש בעומס עבודה

<span id="usage"></span>
שימוש הוא מדידה של כמות הזמן שה-QPU נעול לצורך עומס העבודה שלך, והוא מחושב בצורה שונה בהתאם למצב ההרצה שבו אתה משתמש.

* שימוש של סשן הוא זמן השעון האמיתי שבו הסשן פעיל. ראה [אורך סשן](/guides/run-jobs-session#session-length) למידע נוסף על מעברי מצב הסשן.
* שימוש של אצווה הוא סכום זמן הקוונטי (הזמן שהקומפלקס של ה-QPU מקדיש לעיבוד העבודה שלך) של כל העבודות באצווה.
* שימוש של עבודה בודדת הוא הזמן הקוונטי שהעבודה משתמשת בו לעיבוד.

שים לב שעבודות שנכשלו או בוטלו נחשבות לשימוש שלך בנסיבות מסוימות - ראה את הסעיף [עבודות שנכשלו ובוטלו](#failed-job) לפרטים.

עבור משתמשי תוכנית בתשלום, השימוש קובע את עלות עומס העבודה. ראה [ניהול עלויות](/guides/manage-cost) לפרטים.

<span id="failed-job"></span>
## שימוש עבור עבודות שנכשלו ובוטלו
כאשר עבודה נכשלת או מבוטלת, השימוש המדווח הוא כדלקמן:

* מצב עבודה בודדת או אצווה: השימוש המדווח הוא הזמן שה-QPU היה נעול לצורך הרצת עומס העבודה שלך עד לרגע הכישלון או הביטול. לכן, אם הכישלון או הביטול אירעו לפני הנעילה, השימוש המדווח הוא אפס. אחרת, השימוש המדווח של עומס העבודה הוא כמות השימוש עד לרגע שבו עומס העבודה נכשל או בוטל. כך, חלק מהעבודות שנכשלו אינן מופיעות בשימוש המדווח שלך ואחרות כן.

* מצב סשן: השימוש המדווח הוא זמן השעון האמיתי שבו הסשן פעיל, ללא קשר למספר העבודות שנכשלות או מבוטלות.

<span id="view-usage"></span>
## שאילתה על השימוש בפועל של עומס עבודה
לאחר שעומס עבודה הסתיים, ישנן מספר דרכים לצפות בשימוש בפועל שלו:

- הרץ [`batch.usage()`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/batch#usage) או [`session.usage()`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/session#usage) ב-`qiskit-ibm-runtime` 0.30 ואילך. אם אתה משתמש בגרסה ישנה יותר של `qiskit-ibm-runtime` (>= 0.23 ו-< 0.30), ניתן עדיין למצוא את השימוש ב-`session.details()["usage_time"]` וב-`batch.details()["usage_time"]`.
- השתמש ב-[`GET /sessions/{id}`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/tags/sessions#tags__sessions__operations__GetSessionDetailsExtendedController_getSessionDetails) כדי לראות את השימוש עבור אצווה או סשן ספציפיים.
- השתמש ב-[`GET /jobs/{id}`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/tags/jobs#tags__jobs__operations__GetJobByIdController_getJobById) כדי לראות את השימוש עבור עבודה בודדת.

<span id="instance-usage"></span>
## צפייה בשימוש של מופע
באפשרותך לצפות בשימוש של מופע בדף [Instances](https://quantum.cloud.ibm.com/instances), או, עבור בעלי ההרשאה המתאימה, בדף [Analytics](https://quantum.cloud.ibm.com/analytics). שים לב שהדפים עשויים להציג מספרי שימוש שונים מכיוון שהם מחשבים את השימוש בצורה שונה.

דף ה-Instances מציג שימוש בזמן אמת עבור 28 הימים האחרונים (גלגלי), עד לשעה הנוכחית ביום הנוכחי. השימוש בדף ה-Analytics מחושב מחדש כל שעה וכולל את 28 הימים המלאים האחרונים; כלומר, הוא מציג שימוש מ-00:00 לפני 28 ימים עד היום, בתחילת השעה.

## אמידת שימוש לפני שליחת עבודה
בעוד שקבלת הערכה מקומית מדויקת מסובכת בשל הפעולות הנוספות שנעשות לדיכוי ולהפחתת שגיאות, תוכל להשתמש בנוסחת הבסיס הבאה כדי לקבל קירוב של השימוש המשוער:

`<per sub-job overhead> + (rep_delay + <circuit length>) * <num executions>`

- `<per sub-job overhead>` הוא תקורה של כ-2 שניות לכל תת-עבודה. זה כולל פעולות כגון טעינת המטען לאלקטרוניקת הבקרה. עבודת הפרימיטיב שלך עשויה להתחלק למספר תת-עבודות אם היא גדולה מדי עבור מנוע ההרצה לעיבוד בבת אחת.
- `rep_delay` הוא אפשרות [הניתנת להתאמה אישית](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-execution-options-v2#rep_delay), וברירת המחדל ניתנת על ידי `backend.default_rep_delay`, שהיא 250 מיקרושניות ברוב ה-Backends של IBM Quantum. שים לב שהורדת `rep_delay` מקטינה את זמן ההרצה הכולל על ה-QPU, אך על חשבון שגיאת הכנת מצב מוגברת; ראה את המדריך [הרצה בקצב חזרה דינמי](/guides/repetition-rate-execution) למידע נוסף.
- `<circuit length>` הוא אורך ההוראות הכולל. כל הוראה לוקחת כמות זמן שונה על ה-QPU, ולכן האורך הכולל משתנה מ-Circuit ל-Circuit. מדידה, לדוגמה, יכולה לקחת 56 פעמים יותר מגייט `x`. ניתן להשתמש ב-`backend.target[<instruction>][<qubit>].duration` כדי למצוא את משך הזמן המדויק לכל הוראה. אורך Circuit טיפוסי הוא ככל הנראה בין 50 ל-100 מיקרושניות. אם אתה משתמש בטכניקות דיכוי או הפחתת שגיאות עם הפרימיטיבים, ייתכן שיוכנסו הוראות נוספות ל-Circuit שלך, מה שיגדיל את אורך ה-Circuit הכולל.
    > **Note:** [האפשרות הניסיונית `scheduler_timing`](/guides/visualize-circuit-timing) מחזירה את זמן ה-Circuit הכולל, אך זה אינו הזמן המשמש לחיוב.
- `<num executions>` הוא המספר הכולל של Circuits כפול מספר הצילומים, כאשר ה-Circuits הם אלו שנוצרו לאחר שידור אלמנטי PUB.
    - אם אתה משתמש בטכניקות הפחתת שגיאות עם הפרימיטיבים, ייתכן שיורצו Circuits נוספים כחלק מתהליך ההפחתה, מה שיגדיל את המספר הכולל של הרצות. בנוסף, טכניקות הפחתת שגיאות מתקדמות כגון PEA ו-PEC כרוכות בתקורה גבוהה הרבה יותר מכיוון שהן מצריכות הרצת Circuits ללמידת רעש.
    - Estimator מקבץ צפי שיתופיים לפי Qubit, מה שמצמצם את מספר ההרצות.

אם אינך משתמש בטכניקות הפחתת שגיאות מתקדמות או `rep_delay` מותאם אישית, תוכל להשתמש ב-`2+0.00035*<num executions>` כנוסחה מהירה.

## השלבים הבאים
> **Tip:** - עיין בטיפים אלה: [צמצם זמן ריצה של עבודה](minimize-time).
>     - הגדר את [זמן ההרצה המרבי](max-execution-time).
>     - למד כיצד לבצע Transpile באופן מקומי בסעיף [Transpile](./transpile/).
>     - נסה את המדריך [השוואת הגדרות Transpiler](/guides/circuit-transpilation-settings).